In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install \
    datasets==2.20.0 \
    pyarrow==16.1.0 \
    fsspec==2024.2.0 \
    huggingface-hub==0.23.4 \
    -q

In [ ]:
import pyarrow
import datasets
import sys

print("Python  :", sys.version)
print("PyArrow :", pyarrow.__version__)
print("Datasets:", datasets.__version__)

# Quick sanity test
try:
    import pyarrow.parquet
    print("PyArrow parquet : OK")
except Exception as e:
    print("PyArrow parquet : FAILED —", e)

In [ ]:
from datasets import load_dataset

massive = load_dataset(
    "AmazonScience/massive",
    "hi-IN",
    split='train',
    trust_remote_code=True
)

massive[0]

In [ ]:
df_massive = pd.DataFrame({
    'text'  : massive['utt'],
    'intent_num': massive['intent']
})
print("Available intents in MASSIVE:")
print(df_massive['intent_num'].dtype)
print(df_massive['intent_num'].unique())

In [ ]:
intent_labels = massive.features['intent'].names
print("All unique intent labels:")
for i, label in enumerate(intent_labels):
    print(f"  {i} : {label}")

In [ ]:
df_massive['intent'] = df_massive['intent_num'].apply(
    lambda x: intent_labels[x]
)

df_massive[['text', 'intent_num', 'intent']].head()

In [ ]:
print(df_massive['intent'].value_counts())

In [ ]:
intent_mapping = {
    # Reminders
    'alarm_set'          : 'set_reminder',

    # Cancel / remove
    'alarm_remove'       : 'cancel',
    'calendar_remove'    : 'cancel',
    'lists_remove'       : 'cancel',

    # Meetings / calendar
    'calendar_set'       : 'schedule_meeting',

    # Retrieve info
    'alarm_query'        : 'retrieve_info',
    'calendar_query'     : 'retrieve_info',
    'datetime_query'     : 'retrieve_info',
    'datetime_convert'   : 'retrieve_info',
    'lists_query'        : 'retrieve_info',

    # Answer query
    'qa_factoid'         : 'answer_query',
    'qa_definition'      : 'answer_query',
    'qa_currency'        : 'answer_query',
    'qa_stock'           : 'answer_query',
    'qa_maths'           : 'answer_query',
    'general_quirky'     : 'answer_query',
    'general_joke'       : 'answer_query',
    'weather_query'      : 'answer_query',

    # Notes
    'lists_createoradd'  : 'save_note',

    # Greet
    'general_greet'      : 'greet',
}

df_filtered = df_massive[
    df_massive['intent'].isin(intent_mapping.keys())
].copy()

df_filtered['intent'] = df_filtered['intent'].map(intent_mapping)
df_filtered = df_filtered[['text', 'intent']]
df_filtered['source'] = 'massive'

print(f"Total rows after filtering: {len(df_filtered)}")
print("\nBasanti intent distribution:")
print(df_filtered['intent'].value_counts())

In [ ]:
print(df_filtered.columns.tolist())

In [ ]:
TARGET_MAX = 700

df_capped = df_filtered.groupby('intent').apply(
    lambda x: x.sample(min(len(x), TARGET_MAX), random_state=42)
).reset_index(drop=True)

print(f"Total rows after capping: {len(df_capped)}")
print("\nBasanti intent distribution:")
print(df_capped['intent'].value_counts())

In [ ]:
print(df_capped.columns.tolist())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/basanti/datasets', exist_ok=True)

In [ ]:
df_capped.to_csv('/content/drive/MyDrive/basanti/datasets/massive_final.csv', index=False)
print(f"Saved {len(df_capped)} rows")
print(df_capped.head())

In [ ]:
from datasets import load_dataset

clinic = load_dataset("clinc_oos", "plus", split='train', trust_remote_code=True)
df_clinic = pd.DataFrame(clinic)

print(df_clinic.head())

In [ ]:
print("Available intents in CLINIC150:")
print(df_clinic['intent'].dtype)
print(df_clinic['intent'].unique())

In [ ]:
intent_labels = clinic.features['intent'].names
print("All unique intent labels in CLINIC150:")
for i, label in enumerate(intent_labels):
    print(f"  {i} : {label}")

df_clinic['intent_str'] = df_clinic['intent'].apply(
    lambda x: intent_labels[x]
)

In [ ]:
df_clinic.head()

In [ ]:
df_clinic.columns = ['text', 'intent_num', 'intent']
df_clinic.head()

In [ ]:
print(df_clinic['intent'].value_counts())

In [ ]:
clinic_mapping = {
    # Reminders
    'reminder'              : 'set_reminder',
    'reminder_update'       : 'set_reminder',
    'alarm'                 : 'set_reminder',
    'timer'                 : 'set_reminder',

    # Meetings
    'calendar'              : 'schedule_meeting',
    'calendar_update'       : 'schedule_meeting',
    'meeting_schedule'      : 'schedule_meeting',
    'schedule_meeting'      : 'schedule_meeting',

    # Notes
    'todo_list'             : 'save_note',
    'todo_list_update'      : 'save_note',
    'shopping_list'         : 'save_note',
    'shopping_list_update'  : 'save_note',

    # Retrieve info
    'date'                  : 'retrieve_info',
    'time'                  : 'retrieve_info',
    'timezone'              : 'retrieve_info',
    'repeat'                : 'retrieve_info',

    # Answer query
    'meaning_of_life'       : 'answer_query',
    'definition'            : 'answer_query',
    'fun_fact'              : 'answer_query',
    'weather'               : 'answer_query',
    'what_song'             : 'answer_query',
    'measurement_conversion': 'answer_query',
    'calculator'            : 'answer_query',

    # Greet
    'greeting'              : 'greet',
    'goodbye'               : 'greet',
    'thank_you'             : 'greet',

    # Cancel
    'cancel'                : 'cancel',
    'no'                    : 'cancel',

    # Out of scope
    'oos'                   : 'out_of_scope',
}

df_clinic_filtered = df_clinic[
    df_clinic['intent'].isin(clinic_mapping.keys())
].copy()

df_clinic_filtered['intent'] = df_clinic_filtered['intent'].map(clinic_mapping)
df_clinic_filtered = df_clinic_filtered[['text', 'intent']]
df_clinic_filtered['source'] = 'clinc150'

print(f"Total rows after filtering: {len(df_clinic_filtered)}")
print(df_clinic_filtered['intent'].value_counts())

In [ ]:
df_clinic_filtered.head()

In [ ]:
df_massive = pd.read_csv('/content/drive/MyDrive/basanti/datasets/massive_final.csv')
df_clinic   = df_clinic_filtered.copy()

df_combined_raw = pd.concat([df_massive, df_clinic], ignore_index=True)

print(f"Total rows: {len(df_combined_raw)}")
print("Combined BEFORE translation (English rows still in CLINC150 part):")
print(df_combined_raw.groupby(['intent'])['source'].value_counts())

In [ ]:
from google.colab import userdata
import os

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token

In [ ]:
!pip install sentencepiece sacremoses -q

In [ ]:
!pip install transformers==4.41.2 -q

In [ ]:
!pip install IndicTransToolkit -q

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_name = "ai4bharat/indictrans2-en-indic-1B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded on {device}")

In [ ]:
from IndicTransToolkit.processor import IndicProcessor

ip = IndicProcessor(inference=True)

In [ ]:
def translate_indictrans(texts, batch_size=16):
    translated = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        # Preprocess
        batch_preprocessed = ip.preprocess_batch(
            batch, src_lang="eng_Latn", tgt_lang="hin_Deva"
        )

        inputs = tokenizer(
            batch_preprocessed,
            truncation=True,
            padding="longest",
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                num_beams=5,
                max_length=128,
            )

        decoded = tokenizer.batch_decode(
            outputs, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )

        # Postprocess
        postprocessed = ip.postprocess_batch(decoded, lang="hin_Deva")
        translated.extend(postprocessed)

        print(f"Translated {min(i+batch_size, len(texts))}/{len(texts)}")

    return translated

In [ ]:
test_sample = df_clinic_filtered['text'].head(5).tolist()
test_results = translate_indictrans(test_sample)

for en, hi in zip(test_sample, test_results):
    print(f"EN: {en}")
    print(f"HI: {hi}")
    print()

In [ ]:
df_clinic_filtered['text_hindi'] = translate_indictrans(df_clinic_filtered['text'].tolist())

print(f"Translated {len(df_clinic_filtered)} rows")
print(df_clinic_filtered[['text', 'text_hindi', 'intent']].sample(10))

In [ ]:
df_clinic_hindi = df_clinic_filtered[['text_hindi', 'intent']].copy()
df_clinic_hindi.columns = ['text', 'intent']
df_clinic_hindi['source'] = 'clinc150'

df_clinic_hindi.to_csv('/content/drive/MyDrive/basanti/datasets/clinc150_hindi.csv', index=False)
print(f"Saved {len(df_clinic_hindi)} rows to Drive")
print(df_clinic_hindi.sample(10))

In [ ]:
df_massive = pd.read_csv('/content/drive/MyDrive/basanti/datasets/massive_final.csv')
df_clinic_hindi = pd.read_csv('/content/drive/MyDrive/basanti/datasets/clinc150_hindi.csv')

df_combined = pd.concat([df_massive, df_clinic_hindi], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=['text'])
df_combined = df_combined.dropna(subset=['text'])

print(f"Total combined rows: {len(df_combined)}")
print(df_combined['intent'].value_counts())

In [ ]:
TARGET_MAX = 700

df_combined_capped = df_combined.groupby('intent', group_keys=False).apply(
    lambda x: x.sample(min(len(x), TARGET_MAX), random_state=42)
).reset_index(drop=True)

print(df_combined_capped['intent'].value_counts())

df_combined_capped.to_csv('/content/drive/MyDrive/basanti/datasets/massive_clinc_combined.csv', index=False)
print("Saved combined dataset!")

In [ ]:
import requests
import pandas as pd

def load_text_from_url(url):
    return requests.get(url).text.strip().split('\n')

# Train split
labels_url = "https://raw.githubusercontent.com/jianguoz/Few-Shot-Intent-Detection/main/Datasets/HWU64/train/label"
seq_url    = "https://raw.githubusercontent.com/jianguoz/Few-Shot-Intent-Detection/main/Datasets/HWU64/train/seq.in"

hwu_labels = load_text_from_url(labels_url)
hwu_utterances = load_text_from_url(seq_url)

print(f"Utterances: {len(hwu_utterances)}")
print(f"Labels: {len(hwu_labels)}")

df_hwu_train = pd.DataFrame({'text': hwu_utterances, 'intent_str': hwu_labels})
print(df_hwu_train.head())
print(f"\nTotal unique intents: {df_hwu_train['intent_str'].nunique()}")
print(df_hwu_train['intent_str'].unique())

In [ ]:
hwu_mapping = {
    # Reminders
    'alarm_set'            : 'set_reminder',

    # Cancel
    'alarm_remove'         : 'cancel',
    'calendar_remove'      : 'cancel',
    'lists_remove'         : 'cancel',
    'general_negate'       : 'cancel',
    'general_dontcare'     : 'cancel',
    'general_commandstop'  : 'cancel',

    # Meetings
    'calendar_set'         : 'schedule_meeting',

    # Retrieve info
    'alarm_query'           : 'retrieve_info',
    'calendar_query'        : 'retrieve_info',
    'datetime_query'        : 'retrieve_info',
    'datetime_convert'      : 'retrieve_info',
    'lists_query'           : 'retrieve_info',
    'general_repeat'        : 'retrieve_info',

    # Answer query
    'qa_factoid'             : 'answer_query',
    'qa_definition'          : 'answer_query',
    'qa_currency'            : 'answer_query',
    'qa_stock'               : 'answer_query',
    'qa_maths'               : 'answer_query',
    'general_quirky'         : 'answer_query',
    'general_joke'           : 'answer_query',
    'general_explain'        : 'answer_query',
    'weather_query'          : 'answer_query',

    # Notes
    'lists_createoradd'      : 'save_note',
}

In [ ]:
df_hwu_filtered = df_hwu_train[
    df_hwu_train['intent_str'].isin(hwu_mapping.keys())
].copy()

df_hwu_filtered['intent'] = df_hwu_filtered['intent_str'].map(hwu_mapping)
df_hwu_filtered = df_hwu_filtered[['text', 'intent']]

print(f"Total rows: {len(df_hwu_filtered)}")
print(df_hwu_filtered['intent'].value_counts())

In [ ]:
df_hwu_filtered['text_hindi'] = translate_indictrans(df_hwu_filtered['text'].tolist())

print(f"Translated {len(df_hwu_filtered)} rows")
print(df_hwu_filtered[['text', 'text_hindi', 'intent']].sample(10))

In [ ]:
df_hwu_hindi = df_hwu_filtered[['text_hindi', 'intent']].copy()
df_hwu_hindi.columns = ['text', 'intent']
df_hwu_hindi['source'] = 'hwu64'

df_hwu_hindi.to_csv('/content/drive/MyDrive/basanti/datasets/hwu64_hindi.csv', index=False)
print(f"Saved {len(df_hwu_hindi)} rows to Drive")
print(df_hwu_hindi.sample(10))

In [ ]:
df_massive = pd.read_csv('/content/drive/MyDrive/basanti/datasets/massive_final.csv')
df_clinc   = pd.read_csv('/content/drive/MyDrive/basanti/datasets/clinc150_hindi.csv')
df_hwu     = pd.read_csv('/content/drive/MyDrive/basanti/datasets/hwu64_hindi.csv')

df_all = pd.concat([df_massive, df_clinc, df_hwu], ignore_index=True)
df_all = df_all.drop_duplicates(subset=['text'])
df_all = df_all.dropna(subset=['text'])

print(f"Total combined rows: {len(df_all)}")
print(df_all['intent'].value_counts())

In [ ]:
print(df_all.groupby(['intent'])['source'].value_counts())

In [ ]:
TARGET_MAX = 700

df_capped = df_all.groupby('intent', group_keys=False).apply(
    lambda x: x.sample(min(len(x), TARGET_MAX), random_state=42)
).reset_index(drop=True)

print(df_capped['intent'].value_counts())

In [ ]:
df_capped.to_csv('/content/drive/MyDrive/basanti/datasets/all_sources_combined.csv', index=False)
print(f"Saved {len(df_capped)} rows")

In [ ]:
greet_and_oos = pd.read_excel('/content/drive/MyDrive/basanti/datasets/greet_&_oos.xlsx')
greet_and_oos.info()

In [ ]:
import pandas as pd

df_massive = pd.read_csv('/content/drive/MyDrive/basanti/datasets/massive_final.csv')
df_clinc   = pd.read_csv('/content/drive/MyDrive/basanti/datasets/clinc150_hindi.csv')
df_hwu     = pd.read_csv('/content/drive/MyDrive/basanti/datasets/hwu64_hindi.csv')
greet_and_oos = pd.read_excel('/content/drive/MyDrive/basanti/datasets/greet_&_oos.xlsx')

df_all = pd.concat([df_massive, df_clinc, df_hwu, greet_and_oos], ignore_index=True)
df_all = df_all.drop_duplicates(subset=['text'])
df_all = df_all.dropna(subset=['text'])

print(f"Total combined rows: {len(df_all)}")
print(df_all['intent'].value_counts())

In [ ]:
print(df_all.groupby(['intent'])['source'].value_counts())

In [ ]:
TARGET_MAX = 700

df_capped = df_all.groupby('intent', group_keys=False).apply(
    lambda x: x.sample(min(len(x), TARGET_MAX), random_state=42)
).reset_index(drop=True)

print(df_capped['intent'].value_counts())

In [ ]:
df_capped.to_csv('/content/drive/MyDrive/basanti/datasets/all_sources_combined.csv', index=False)
print(f"Saved {len(df_capped)} rows")

In [ ]:
df_capped['intent'].value_counts()